# COMP 6940: Big Data and Data Visualization

### Project: Airline Delay Analysis and Performance Optimization  
#### Dataset: Carrier On-Time Performance Dataset  

## **Notebook:** 01 — Data Cleaning and Preprocessing  

**Objective:**  

This notebook focuses on preparing the raw dataset for analysis by performing data cleaning, transformation, and         feature engineering to produce a structured, analysis-ready dataset.


---

## 01 — Data Cleaning and Preprocessing

This section outlines the data cleaning and preprocessing steps applied to the Carrier On-Time Performance dataset. The objective of this stage is to transform the raw dataset into a structured and analysis-ready format suitable for delay attribution and further downstream statistical analysis.

Key preprocessing steps include handling missing values, standardizing variable formats, converting date and time fields, and generating new features required for analysis.

The final cleaned dataset is exported as `cleaned_flight_data.csv`, which will be used in subsequent stages of the project.

In [1]:
import numpy as np
import pandas as pd

### Load Raw Data

The raw dataset is imported into the environment for preprocessing. This dataset contains flight-level information including scheduling details, delays, and delay causes.

At this stage, no transformations are applied yet; the purpose is to inspect the structure, data types, and initial quality of the dataset before cleaning.

In [2]:
df = pd.read_csv("airline_2m.csv", encoding="ISO-8859-1", low_memory=False)
df.shape

(2000000, 109)

### Initial Data Quality Audit

Before cleaning the dataset, the structure, data types, missing values, and duplicate records are examined. This provides a baseline understanding of the raw dataset and helps justify the preprocessing decisions made later in the notebook.

In [3]:
print("Dataset shape:", df.shape)

display(df.head())
display(df.info())

missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percent", ascending=False)

display(missing_summary.head(20))

print("Duplicate rows:", df.duplicated().sum())

Dataset shape: (2000000, 109)


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div4WheelsOff,Div4TailNum,Div5Airport,Div5AirportID,Div5AirportSeqID,Div5WheelsOn,Div5TotalGTime,Div5LongestGTime,Div5WheelsOff,Div5TailNum
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Columns: 109 entries, Year to Div5TailNum
dtypes: float64(72), int64(18), str(19)
memory usage: 1.8 GB


None

,missing_count,missing_percent
Div4WheelsOn,2000000,100.0
Div4TotalGTime,2000000,100.0
Div4LongestGTime,2000000,100.0
Div4WheelsOff,2000000,100.0
Div4TailNum,2000000,100.0
Div5Airport,2000000,100.0
Div5AirportID,2000000,100.0
Div5AirportSeqID,2000000,100.0
Div5WheelsOn,2000000,100.0
Div5TotalGTime,2000000,100.0


Duplicate rows: 0


### Drop Empty Columns and Normalize Delay Cause Variables

Columns that contain entirely missing or irrelevant values are removed to reduce noise and improve computational efficiency.

Additionally, missing values in delay cause variables are standardized to ensure consistency in later calculations. This step is important because delay-related metrics will be aggregated and analyzed, and inconsistent missing values could distort results.

In [4]:
initial_rows = df.shape[0]
initial_cols = df.shape[1]

df = df.dropna(how="all", axis=1)

removed_cols = initial_cols - df.shape[1]
print(f"Removed {removed_cols} completely empty columns.")

delay_cause_cols = [
    "CarrierDelay",
    "WeatherDelay",
    "NASDelay",
    "SecurityDelay",
    "LateAircraftDelay",
]

for col in delay_cause_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

if "CancellationCode" in df.columns:
    df["CancellationCode"] = df["CancellationCode"].fillna("unknown")

print("Dataset shape after dropping empty columns:", df.shape)

Removed 24 completely empty columns.
Dataset shape after dropping empty columns: (2000000, 85)


### Missing Value Treatment Check

Missing delay cause values were filled with zero because flights without a recorded delay cause are treated as having no delay attributed to that specific category. Cancellation codes were filled with `"unknown"` so that cancelled flights without a specific cancellation reason are still retained in the dataset.

In [5]:
delay_missing_after = df[delay_cause_cols].isna().sum()
display(delay_missing_after)

if "CancellationCode" in df.columns:
    print(df["CancellationCode"].value_counts(dropna=False).head())

CarrierDelay         0
WeatherDelay         0
NASDelay             0
SecurityDelay        0
LateAircraftDelay    0
dtype: int64

CancellationCode
unknown    1979780
B             8469
A             7004
C             3749
D              998
Name: count, dtype: int64


### Date and Categorical Variable Processing

Date-related variables are converted into appropriate datetime formats to enable time-based analysis. Categorical variables are also standardized to ensure consistent labeling across the dataset.

This step ensures that temporal trends (e.g., daily or monthly patterns) and categorical comparisons can be accurately computed in later stages.

In [6]:
df["FlightDate"] = pd.to_datetime(df["FlightDate"], errors="coerce")

invalid_dates = df["FlightDate"].isna().sum()
print("Invalid or missing FlightDate values:", invalid_dates)

for col in (
    "Reporting_Airline",
    "IATA_CODE_Reporting_Airline",
    "Origin",
    "Dest",
    "CancellationCode",
):
    if col in df.columns:
        df[col] = df[col].astype("category")

df.head()

Invalid or missing FlightDate values: 0


,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### State Code Standardization

State codes are converted to uppercase and stored using a string data type. This approach preserves missing values while ensuring consistency in formatting.

Standardizing state codes is essential for grouping, filtering, and geographic analysis in later stages of the project.

In [7]:
for col in ("OriginState", "DestState"):
    if col in df.columns:
        df[col] = df[col].astype("string").str.upper()

df[["OriginState","DestState"]]

,OriginState,DestState
0,MN,UT
1,WI,FL
2,CO,TX
3,CA,MI
4,NJ,NC
...,...,...
1999995,NV,AZ
1999996,NJ,TX
1999997,SC,NC
1999998,IL,TN


### Scheduled Time Conversion (HHMM to Minutes)

Scheduled departure and arrival times, originally stored in HHMM format, are converted into minutes since midnight. This transformation allows for easier numerical analysis and comparison of time-based variables.

Invalid or missing time values are converted to `NaN` to prevent incorrect calculations.

Arrival and departure delay variables are retained in their original form, including negative values, which represent early arrivals or departures. It was important to preserve these values in order to accurately capture flight performance.

In [8]:
def hhmm_to_minutes(series):
    v = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float, copy=False)
    out = np.full(v.shape, np.nan, dtype=float)
    mask = np.isfinite(v) & (v >= 0)
    vi = np.floor(v[mask]).astype(np.int64, copy=False)
    out[mask] = (vi // 100) * 60 + (vi % 100)
    return pd.Series(out, index=series.index)


time_cols = [
    "CRSDepTime",
    "DepTime",
    "WheelsOff",
    "WheelsOn",
    "CRSArrTime",
    "ArrTime",
]
for col in time_cols:
    if col in df.columns:
        df[col] = hhmm_to_minutes(df[col])

df

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,Div1WheelsOff,Div1TailNum,Div2Airport,Div2AirportID,Div2AirportSeqID,Div2WheelsOn,Div2TotalGTime,Div2LongestGTime,Div2WheelsOff,Div2TailNum
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1999995,2008,1,3,23,7,2008-03-23,WN,19393,WN,N712SW,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999996,1999,1,1,5,2,1999-01-05,CO,19704,CO,N14308,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999997,2003,4,11,14,5,2003-11-14,US,20355,US,N528AU,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1999998,2012,2,5,15,2,2012-05-15,WN,19393,WN,N281WN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Time Conversion Validation

Scheduled and actual time fields were converted from HHMM format into minutes after midnight. The converted values are checked to ensure they fall within a valid daily range from 0 to 1439 minutes.

In [9]:
for col in time_cols:
    if col in df.columns:
        invalid_time_count = ((df[col] < 0) | (df[col] > 1439)).sum()
        print(f"{col}: {invalid_time_count} invalid converted time values")

CRSDepTime: 1 invalid converted time values
DepTime: 161 invalid converted time values
WheelsOff: 157 invalid converted time values
WheelsOn: 407 invalid converted time values
CRSArrTime: 57 invalid converted time values
ArrTime: 713 invalid converted time values


### Calendar Features and Seasonal Classification

Additional calendar-based features are derived from the `FlightDate` and `Month` variables. These include attributes such as day, month, and season.

Seasonal classification enables the analysis of trends across different times of the year, which is particularly relevant for identifying patterns in flight delays.

In [10]:
df["Year"] = df["FlightDate"].dt.year
df["Quarter"] = df["FlightDate"].dt.quarter
df["Month"] = df["FlightDate"].dt.month
df["DayOfWeek"] = df["FlightDate"].dt.dayofweek + 1


def month_to_season(m):
    if m in (12, 1, 2):
        return "Winter"
    if m in (3, 4, 5):
        return "Spring"
    if m in (6, 7, 8):
        return "Summer"
    return "Autumn"


df["Season"] = df["Month"].map(month_to_season).astype("category")
df[["Month","Season"]]

,Month,Season
0,1,Winter
1,5,Spring
2,6,Summer
3,8,Summer
4,1,Winter
...,...,...
1999995,3,Spring
1999996,1,Winter
1999997,11,Autumn
1999998,5,Spring


In [11]:
display(df[["Year", "Quarter", "Month", "DayOfWeek", "Season"]].head())

print("Month range:", df["Month"].min(), "to", df["Month"].max())
print("DayOfWeek range:", df["DayOfWeek"].min(), "to", df["DayOfWeek"].max())

display(df["Season"].value_counts())

,Year,Quarter,Month,DayOfWeek,Season
0,1998,1,1,5,Winter
1,2009,2,5,4,Spring
2,2013,2,6,6,Summer
3,2010,3,8,2,Summer
4,2006,1,1,7,Winter


Month range: 1 to 12
DayOfWeek range: 1 to 7


Season
Summer    509872
Spring    505075
Autumn    492944
Winter    492109
Name: count, dtype: int64

### Delay Indicators and Derived Features

New variables are created to enhance the analysis of flight delays. These include:

- Binary indicators identifying whether a flight experienced a delay
- Aggregated delay cause totals
- Flags for specific delay conditions
- Route-level identifiers
- Time-based features such as departure and arrival hours

These derived features allow for more detailed analysis of delay patterns, including identifying dominant delay causes and examining trends across routes and time periods.

In [12]:
df["IsArrivalDelayed"] = df["ArrDelay"] > 0
df["IsDepartureDelayed"] = df["DepDelay"] > 0

df["TotalCauseDelay"] = df[delay_cause_cols].sum(axis=1)

df["HasCarrierDelay"] = df["CarrierDelay"] > 0
df["HasWeatherDelay"] = df["WeatherDelay"] > 0
df["HasNASDelay"] = df["NASDelay"] > 0
df["HasSecurityDelay"] = df["SecurityDelay"] > 0
df["HasLateAircraftDelay"] = df["LateAircraftDelay"] > 0

df["Route"] = df["Origin"].astype(str) + "-" + df["Dest"].astype(str)

df["DepHour"] = (df["CRSDepTime"] // 60).astype("Int64")
df["ArrHour"] = (df["CRSArrTime"] // 60).astype("Int64")

df.head()

,Year,Quarter,Month,DayofMonth,DayOfWeek,FlightDate,Reporting_Airline,DOT_ID_Reporting_Airline,IATA_CODE_Reporting_Airline,Tail_Number,...,IsDepartureDelayed,TotalCauseDelay,HasCarrierDelay,HasWeatherDelay,HasNASDelay,HasSecurityDelay,HasLateAircraftDelay,Route,DepHour,ArrHour
0,1998,1,1,2,5,1998-01-02,NW,19386,NW,N297US,...,True,0.0,False,False,False,False,False,MSP-SLC,16,18
1,2009,2,5,28,4,2009-05-28,FL,20437,FL,N946AT,...,False,0.0,False,False,False,False,False,MKE-MCO,12,15
2,2013,2,6,29,6,2013-06-29,MQ,20398,MQ,N665MQ,...,True,0.0,False,False,False,False,False,GJT-DFW,16,19
3,2010,3,8,31,2,2010-08-31,DL,19790,DL,N6705Y,...,False,0.0,False,False,False,False,False,LAX-DTW,13,20
4,2006,1,1,15,7,2006-01-15,US,20355,US,N504AU,...,True,32.0,False,False,False,False,True,EWR-CLT,18,20


### Delay Variable and Outlier Audit

Delay variables are inspected to identify extreme values and confirm that early flights, on-time flights, and delayed flights are represented correctly. Extreme delays are retained because they may represent real operational disruptions, but they are documented so they can be interpreted carefully in later analysis.

In [13]:
delay_cols = ["ArrDelay", "DepDelay", "CarrierDelay", "WeatherDelay", 
              "NASDelay", "SecurityDelay", "LateAircraftDelay", "TotalCauseDelay"]

existing_delay_cols = [col for col in delay_cols if col in df.columns]

display(df[existing_delay_cols].describe().T)

for col in ["ArrDelay", "DepDelay"]:
    if col in df.columns:
        q99 = df[col].quantile(0.99)
        max_val = df[col].max()
        print(f"{col}: 99th percentile = {q99}, max = {max_val}")

,count,mean,std,min,25%,50%,75%,max
ArrDelay,1958922.0,6.205467,34.833399,-706.0,-10.0,-1.0,10.0,1898.0
DepDelay,1963932.0,8.587405,32.724728,-990.0,-3.0,0.0,7.0,1878.0
CarrierDelay,2000000.0,1.873412,16.281192,0.0,0.0,0.0,0.0,1878.0
WeatherDelay,2000000.0,0.326043,7.087432,0.0,0.0,0.0,0.0,1847.0
NASDelay,2000000.0,1.706707,11.259688,0.0,0.0,0.0,0.0,1343.0
SecurityDelay,2000000.0,0.009413,0.702990,0.0,0.0,0.0,0.0,219.0
LateAircraftDelay,2000000.0,2.445841,15.497416,0.0,0.0,0.0,0.0,1407.0
TotalCauseDelay,2000000.0,6.361415,27.659285,0.0,0.0,0.0,0.0,1898.0


ArrDelay: 99th percentile = 148.0, max = 1898.0
DepDelay: 99th percentile = 144.0, max = 1878.0


### Data Validation

Basic validation checks performed to ensure that the preprocessing steps were applied correctly. This includes verifying data types, checking for unexpected missing values, and confirming that derived variables behave as expected.

These checks help ensure data integrity before proceeding to analysis.

In [14]:
validation_checks = {
    "FlightDate has valid dates": df["FlightDate"].notna().all(),
    "Season has no missing values": df["Season"].notna().all(),
    "Arrival delay flag is boolean": df["IsArrivalDelayed"].dtype == bool,
    "Departure delay flag is boolean": df["IsDepartureDelayed"].dtype == bool,
    "TotalCauseDelay is non-negative": (df["TotalCauseDelay"] >= 0).all(),
    "DepHour values are valid": df["DepHour"].dropna().between(0, 23).all(),
    "ArrHour values are valid": df["ArrHour"].dropna().between(0, 23).all(),
}

for check, result in validation_checks.items():
    print(f"{check}: {result}")

df[
    [
        "FlightDate",
        "Season",
        "IsArrivalDelayed",
        "IsDepartureDelayed",
        "TotalCauseDelay",
        "Route",
        "DepHour",
        "ArrHour",
    ]
].head()

FlightDate has valid dates: True
Season has no missing values: True
Arrival delay flag is boolean: True
Departure delay flag is boolean: True
TotalCauseDelay is non-negative: True
DepHour values are valid: False
ArrHour values are valid: False


,FlightDate,Season,IsArrivalDelayed,IsDepartureDelayed,TotalCauseDelay,Route,DepHour,ArrHour
0,1998-01-02,Winter,True,True,0.0,MSP-SLC,16,18
1,2009-05-28,Spring,False,False,0.0,MKE-MCO,12,15
2,2013-06-29,Summer,False,True,0.0,GJT-DFW,16,19
3,2010-08-31,Summer,False,False,0.0,LAX-DTW,13,20
4,2006-01-15,Winter,True,True,32.0,EWR-CLT,18,20


### Final Cleaned Dataset Summary

The final dataset is summarized before export to confirm the number of records, number of variables, remaining missing values, and key engineered features. This ensures that the cleaned dataset is ready for the downstream exploratory analysis, delay attribution, and clustering notebooks.

In [15]:
print("Final dataset shape:", df.shape)
print("Total remaining missing values:", df.isna().sum().sum())

engineered_features = [
    "Season",
    "IsArrivalDelayed",
    "IsDepartureDelayed",
    "TotalCauseDelay",
    "Route",
    "DepHour",
    "ArrHour"
]

print("Engineered features created:")
for feature in engineered_features:
    print("-", feature)

display(df[engineered_features].head())

Final dataset shape: (2000000, 97)
Total remaining missing values: 50126999
Engineered features created:
- Season
- IsArrivalDelayed
- IsDepartureDelayed
- TotalCauseDelay
- Route
- DepHour
- ArrHour


,Season,IsArrivalDelayed,IsDepartureDelayed,TotalCauseDelay,Route,DepHour,ArrHour
0,Winter,True,True,0.0,MSP-SLC,16,18
1,Spring,False,False,0.0,MKE-MCO,12,15
2,Summer,False,True,0.0,GJT-DFW,16,19
3,Summer,False,False,0.0,LAX-DTW,13,20
4,Winter,True,True,32.0,EWR-CLT,18,20


### Save Cleaned Dataset

The fully processed dataset is saved as `cleaned_flight_data.csv`. This file serves as the input for subsequent analysis and modeling stages.

Saving the cleaned dataset ensures reproducibility and allows future steps to be performed without repeating preprocessing operations.

In [16]:
df.to_csv("cleaned_flight_data.csv", index=False)